Add Metadata including YEAR to coords provided by Dr Guyot

In [ ]:
import pandas as pd

In [ ]:
subset = pd.read_csv(r"../..\data\dr_guyot_all_for_collection.csv")

print("subset shape:", subset.shape)

subset shape: (3783, 3)


In [ ]:
gbif = pd.read_csv(r"../../../../example\gbif_example\0026013-240906103802322\0026013-240906103802322.csv", delimiter="\t")
print("gbif shape:", gbif.shape)

gbif shape: (33668, 50)


In [11]:
required_subset_cols = ["latitude", "longitude"]
required_gbif_cols = ["decimalLatitude", "decimalLongitude"]

missing_subset = [c for c in required_subset_cols if c not in subset.columns]
missing_gbif = [c for c in required_gbif_cols if c not in gbif.columns]

if missing_subset:
    raise KeyError(f"subset.csv is missing: {missing_subset}")
if missing_gbif:
    raise KeyError(f"FullData.csv is missing: {missing_gbif}")

print("OK: required coordinate columns found.")


OK: required coordinate columns found.


In [12]:

subset_keyed = subset.copy()
gbif_keyed = gbif.copy()

subset_keyed["_key"] = subset_keyed["latitude"].astype(str) + "|" + subset_keyed["longitude"].astype(str)
gbif_keyed["_key"] = gbif_keyed["decimalLatitude"].astype(str) + "|" + gbif_keyed["decimalLongitude"].astype(str)

print("Keys created.")
print("Example subset key:", subset_keyed["_key"].iloc[0] if len(subset_keyed) else "subset empty")
print("Example gbif key:", gbif_keyed["_key"].iloc[0] if len(gbif_keyed) else "gbif empty")


Keys created.
Example subset key: -24.36666|46.45
Example gbif key: 0.0|0.0


In [13]:
# Cell 6 — (Optional) Quick diagnostic: how many GBIF rows have a key present in subset?
subset_keys = set(subset_keyed["_key"])
gbif_in_subset = gbif_keyed["_key"].isin(subset_keys).sum()

print("GBIF rows whose (lat,lon) key exists in subset:", gbif_in_subset)


GBIF rows whose (lat,lon) key exists in subset: 8831


In [14]:
# Cell 7 — OUTPUT #1: Expanded match table (one row per GBIF record matched; subset rows repeat)
# Keep ALL subset columns + a chosen set of GBIF metadata columns (including year)
gbif_keep = [
    "year",
    "eventDate",
    "species",
    "scientificName",
    "gbifID",
    "occurrenceID",
    "basisOfRecord",
    "country",
    "stateProvince",
    "locality",
]

gbif_keep_available = [c for c in gbif_keep if c in gbif_keyed.columns]

expanded = subset_keyed.merge(
    gbif_keyed[["_key"] + gbif_keep_available],
    on="_key",
    how="left",
    suffixes=("", "_gbif"),
)

expanded_no_key = expanded.drop(columns=["_key"])
expanded_no_key.to_csv("subset_gbif_expanded.csv", index=False)

print("Wrote: subset_gbif_expanded.csv")
print("Expanded shape:", expanded_no_key.shape)


Wrote: subset_gbif_expanded.csv
Expanded shape: (14593, 12)


In [15]:
# Cell 8 — OUTPUT #2: Enriched subset table (same rows as subset; add match counts + year summary)
agg = gbif_keyed.groupby("_key", dropna=False).agg(
    n_gbif_rows=("year", "size"),
    n_years=("year", lambda s: s.dropna().nunique()),
    min_year=("year", "min"),
    max_year=("year", "max"),
    years_list=("year", lambda s: ",".join(str(int(y)) for y in sorted(s.dropna().unique()))),
).reset_index()

enriched = subset_keyed.merge(agg, on="_key", how="left")

# Fill "no match" rows cleanly (subset rows preserved)
enriched["n_gbif_rows"] = enriched["n_gbif_rows"].fillna(0).astype(int)
enriched["n_years"] = enriched["n_years"].fillna(0).astype(int)

enriched_no_key = enriched.drop(columns=["_key"])
enriched_no_key.to_csv("subset_enriched.csv", index=False)

print("Wrote: subset_enriched.csv")
print("Enriched shape:", enriched_no_key.shape)


Wrote: subset_enriched.csv
Enriched shape: (3783, 8)


In [16]:
# Cell 9 — Sanity checks
print("Subset rows:", len(subset))
print("Enriched rows (should match subset):", len(enriched_no_key))

print("\nTop 10 subset points by number of GBIF matches:")
display(
    enriched_no_key.sort_values("n_gbif_rows", ascending=False).head(10)
)


Subset rows: 3783
Enriched rows (should match subset): 3783

Top 10 subset points by number of GBIF matches:


,specimen_id,latitude,longitude,n_gbif_rows,n_years,min_year,max_year,years_list
2447,Coffea_mauritiana,-21.306270,55.641980,239,14,1998.0,2017.0,"1998,1999,2001,2002,2003,2005,2006,2007,2010,2..."
2830,Coffea_montis-sacri,-21.383333,47.866667,175,1,1999.0,1999.0,1999
3733,Coffea_vianneyi,-21.383333,47.866667,175,1,1999.0,1999.0,1999
2847,Coffea_moratii,-21.383333,47.866667,175,1,1999.0,1999.0,1999
98,Coffea_bertrandii,-21.383333,47.866667,175,1,1999.0,1999.0,1999
2670,Coffea_millotii,-21.383333,47.866667,175,1,1999.0,1999.0,1999
3729,Coffea_vatovavyensis,-21.383333,47.866667,175,1,1999.0,1999.0,1999
914,Coffea_coursiana,-21.383333,47.866667,175,1,1999.0,1999.0,1999
372,Coffea_buxifolia,-21.383333,47.866667,175,1,1999.0,1999.0,1999
1242,Coffea_farafanganensis,-21.383333,47.866667,175,1,1999.0,1999.0,1999
